In [ ]:
import numpy
import pandas

import matplotlib
import matplotlib.pyplot as plt
plt.style.use('../mystyle.mplstyle')

import sbruceana

PATH_TO_SBRUCE = "/Users/triozzi/Analysis/numine/sbruceana/data/cc1e0pi/"

#### No cut at all

In [ ]:
FILE_CV = "nocut/CNAF_CV_1eNp0pi_NuMI_NoSysts_NoCut.root"
FILE_OFFBEAM = "nocut/CNAF_OffBeam_1eNp0pi_NuMI_NoSysts_NoCut.root"
FILE_DATA = "nocut/CNAF_Data_1eNp0pi_NuMI_NoSysts_NoCut.root"

FILE_DIRT = "nocut/CNAF_Dirt_1eNp0pi_NuMI_NoCut.root"

In [ ]:
# MC
df = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedNu"  
)
pot = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")
time = sbruceana.utils.get_livetime_data(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedCos"  
)
pot_cos = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos['cosmic'] = 1
df = pandas.concat(
  (df, df_cos)
)

# data
df_data = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_DATA}",
  "events/selectedData"  
)
pot_data = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_DATA}")
time_data = sbruceana.utils.get_livetime_data(f"{PATH_TO_SBRUCE}{FILE_DATA}")

# offbeam
df_offbeam = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_OFFBEAM}",
  "offbeam/selectedOffbeam"  
)
time_offbeam = sbruceana.utils.get_livetime_offbeam(f"{PATH_TO_SBRUCE}{FILE_OFFBEAM}")

# dirt
df_dirt = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_DIRT}",
  "events/selectedNu"  
)
pot_dirt = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_DIRT}")
time_dirt = sbruceana.utils.get_livetime_data(f"{PATH_TO_SBRUCE}{FILE_DIRT}")

In [ ]:
(time_data  - (pot_data/pot) * time) / time_offbeam, time_data/time_offbeam

In [ ]:
(time_data  - (pot_data/pot) * time) / time_offbeam

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

# w/o dirts
var = "Slice"
width = 1; bins = numpy.arange(0, 30+width, width)

y_MC, bins = numpy.histogram(df[var], bins=bins)
x = 0.5 * (bins[1:] - bins[:1])
ax.scatter(x, y_MC, label='MC nu + cosmics')

y_ob, bins = numpy.histogram(df_offbeam[var], bins=bins)
x = 0.5 * (bins[1:] - bins[:1])
ax.scatter(x, y_ob, label='off-beam cosmics')

y_data, bins = numpy.histogram(df_data[var], bins=bins)
x = 0.5 * (bins[1:] - bins[:1])
ax.scatter(x, y_data, label='data', c='black')

x = 0.5 * (bins[1:] - bins[:1])
ax.scatter(x, pot_data/pot * y_MC + time_data/time_offbeam*y_ob, label='sim\npot_data/pot_MC * MC +\ntime_data/time_offbeam * offbeam', c='red')

ax.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

# w/ dirts
var = "Slice"
width = 1; bins = numpy.arange(0, 30+width, width)

y_MC, bins = numpy.histogram(df[var], bins=bins)
x = 0.5 * (bins[1:] - bins[:1])
ax.scatter(x, y_MC, label='MC nu + cosmics')

y_dirt, bins = numpy.histogram(df_dirt[var], bins=bins)
x = 0.5 * (bins[1:] - bins[:1])
ax.scatter(x, y_dirt, label='MC dirts (nu-only)')

y_ob, bins = numpy.histogram(df_offbeam[var], bins=bins)
x = 0.5 * (bins[1:] - bins[:1])
ax.scatter(x, y_ob, label='off-beam cosmics')

y_data, bins = numpy.histogram(df_data[var], bins=bins)
x = 0.5 * (bins[1:] - bins[:1])
ax.scatter(x, y_data, label='data', c='black')

x = 0.5 * (bins[1:] - bins[:1])
ax.scatter(x, pot_data/pot * (y_MC + y_dirt * pot/pot_dirt ) + time_data/time_offbeam*y_ob, label='sim\npot_data/pot_MC * MC +\ntime_data/time_offbeam * offbeam', c='red')

ax.legend()

In [ ]:
print('pot:          ', pot)
print('pot_data:     ', pot_data)
print('time_data:    ', time_data)
print('time_offbeam: ', time_offbeam)
print('time_MC:      ', time)
print('y_data.sum(): ', y_data.sum())
print('y_mc.sum():   ', y_MC.sum())
print('y_ob.sum():   ', y_ob.sum())

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "Slice"
width = 1; bins = numpy.arange(0, 20+width, width)

# ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=False, band=True, clip=False)
# ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, False)
IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED)

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'slice index [#]',
  ylabel = f'slices [{tag_count}]\n/ {width}',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=9); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/nocut/nocut_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "trueE"
width = 0.2; bins = numpy.arange(0, 10+width, width)

IS_AREA_NORMALIZED = False
ax = sbruceana.plotting.plot_by_category(ax, df_dirt, sbruceana.config.GENERIC_NU_CATEGORIES, bins, var, 1, True)

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI dirt MC: {pot_dirt:.1e} POT',
  xlabel = '$E_{\\nu}$ [GeV]',
  ylabel = f'slices [{tag_count}]\n/ {width}',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=9, title='no selection cut'); leg.get_title().set_fontsize(11)

fig.savefig(f"plots/nocut/nocut_dirtonly_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "vtxx"
width = 10; bins = numpy.arange(-380, 380+width, width)

IS_AREA_NORMALIZED = False
ax = sbruceana.plotting.plot_by_category(ax, df_dirt, sbruceana.config.GENERIC_NU_CATEGORIES, bins, var, 1, True)

ax = sbruceana.plotting.place_cut(ax, -335, False)
ax = sbruceana.plotting.place_cut(ax, 335, True)
ax = sbruceana.plotting.place_window(ax, [-85, 85], True)

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI dirt MC: {pot_dirt:.1e} POT',
  xlabel = 'reco. vertex $x$ [cm]',
  ylabel = f'slices [{tag_count}]\n/ {width} cm',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=9, title='no selection cut'); leg.get_title().set_fontsize(11)

fig.savefig(f"plots/nocut/nocut_dirtonly_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "vtxy"
width = 5; bins = numpy.arange(-210, 160+width, width)

IS_AREA_NORMALIZED = False
ax = sbruceana.plotting.plot_by_category(ax, df_dirt, sbruceana.config.GENERIC_NU_CATEGORIES, bins, var, 1, True)

ax = sbruceana.plotting.place_cut(ax, -168, False)
ax = sbruceana.plotting.place_cut(ax, 115, True)

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI dirt MC: {pot_dirt:.1e} POT',
  xlabel = 'reco. vertex $y$ [cm]',
  ylabel = f'slices [{tag_count}]\n/ {width} cm',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=9, title='no selection cut'); leg.get_title().set_fontsize(11)

fig.savefig(f"plots/nocut/nocut_dirtonly_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "vtxz"
width = 20; bins = numpy.arange(-1000, 1000+width, width)

IS_AREA_NORMALIZED = False
ax = sbruceana.plotting.plot_by_category(ax, df_dirt, sbruceana.config.GENERIC_NU_CATEGORIES, bins, var, 1, True)

ax = sbruceana.plotting.place_cut(ax, -867, False)
ax = sbruceana.plotting.place_cut(ax, 845, True)

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI dirt MC: {pot_dirt:.1e} POT',
  xlabel = 'reco. vertex $z$ [cm]',
  ylabel = f'slices [{tag_count}]\n/ {width} cm',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=9, title='no selection cut'); leg.get_title().set_fontsize(11)

fig.savefig(f"plots/nocut/nocut_dirtonly_{var}.pdf", dpi=300)

In [ ]:
mask_fv = (
    numpy.isfinite(df_dirt["vtxx"]) &
    numpy.isfinite(df_dirt["vtxy"]) &
    numpy.isfinite(df_dirt["vtxz"]) &
    (
        (
            (
                (df_dirt["vtxx"] < -61.94 - 25) &
                (df_dirt["vtxx"] > -358.49 + 25)
            ) |
            (
                (df_dirt["vtxx"] > 61.94 + 25) &
                (df_dirt["vtxx"] < 358.49 - 25)
            )
        ) &
        (
            (df_dirt["vtxy"] > -181.86 + 25) &
            (df_dirt["vtxy"] < 134.96 - 25)
        ) &
        (
            (df_dirt["vtxz"] > -894.95 + 30) &
            (df_dirt["vtxz"] < 894.95 - 50)
        )
    )
)

In [ ]:
1 - (len(df_dirt[mask_fv]) / len(df_dirt))

In [ ]:
def draw_numi_line(ax, dx, dy, anchor_x=0., anchor_y=0., label='NuMI dir'):
    """
    Parametric line through (anchor_x, anchor_y) with slope (dx, dy).
    Extended to fill the full axes range. Arrow tip at forward end.
    """
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    ts = []
    for d, lo, hi, c in [(dx, xmin, xmax, anchor_x),
                          (dy, ymin, ymax, anchor_y)]:
        if abs(d) > 1e-9:
            ts += [(lo - c) / d, (hi - c) / d]

    t_neg = max(t for t in ts if t <= 0)
    t_pos = min(t for t in ts if t >= 0)

    x0, y0 = anchor_x + t_neg * dx, anchor_y + t_neg * dy
    x1, y1 = anchor_x + t_pos * dx, anchor_y + t_pos * dy

    ax.plot([x0, x1], [y0, y1], color='red', lw=0.75, label=label, zorder=5)
    # Arrow tip at forward end (positive t direction = beam forward)
    ax.annotate('',
                xy     =(x1, y1),
                xytext =(x1 - numpy.sign(t_pos) * dx * 1e-3,
                         y1 - numpy.sign(t_pos) * dy * 1e-3),
                arrowprops=dict(arrowstyle='->', color='red', lw=0.75),
                zorder=6)
    return

In [ ]:
fig, axes = plt.subplots(figsize=(4.25*2, 3.5), ncols=2, layout='constrained')

# AV boundaries
av_rect_kwargs = dict(linewidth=0.75, edgecolor='black', facecolor='none', linestyle='-', zorder=3)
# NuMI direction
numi_dx, numi_dy, numi_dz = 3.94583e-01, 4.26067e-02, 9.17677e-01

# common normalization
h_xy, xedges_xy, yedges_xy = numpy.histogram2d(
    df_dirt.truevtxx, df_dirt.truevtxy,
    bins=(numpy.arange(-1.5e3, 1.5e3+10, 10),
          numpy.arange(-1.e3,  1.25e3+10, 10)),
)
h_zy, xedges_zy, yedges_zy = numpy.histogram2d(
    df_dirt.truevtxz, df_dirt.truevtxy,
    bins=(numpy.arange(-2.5e3, 2.5e3+10, 10),
          numpy.arange(-1.e3,  1.25e3+10, 10)),
)
vmin = 1  # log norm: avoid 0
vmax = max(h_xy.max(), h_zy.max())
norm = matplotlib.colors.LogNorm(vmin=vmin, vmax=vmax)

##### XY
ax = axes[0]
h2d_xy = ax.hist2d(
  df_dirt.truevtxx, df_dirt.truevtxy,
  bins = (
    numpy.arange(-1.5e3, 1.5e3+10, 10),
    numpy.arange(-1.e3, 1.25e3+10, 10),
  ),
  norm=norm,
  cmap = 'coolwarm',
  rasterized = True
)
# AV
for x0, x1 in [(-358.49, -61.94), (61.94, 358.49)]:
    ax.add_patch(matplotlib.patches.Rectangle(
        (x0, -181.86), x1 - x0, 134.96 - (-181.86), **av_rect_kwargs
    ))
# NuMI dir
draw_numi_line(ax, numi_dx, numi_dy, anchor_x=0., anchor_y=0.)

ax.set(
  xlabel = 'true vertex $x$ / 10 cm',
  ylabel = 'true vertex $y$ / 10 cm',
)

##### ZY
ax = axes[1]
h2d_zy = ax.hist2d(
  df_dirt.truevtxz, df_dirt.truevtxy,
  bins = (
    numpy.arange(-2.5e3, 2.5e3+10, 10),
    numpy.arange(-1.e3, 1.25e3+10, 10),
  ),
  norm=norm,
  cmap = 'coolwarm',
  rasterized = True
)
ax.set(
  title = f'NuMI dirt MC: {pot_dirt:.1e} POT',
  xlabel = 'true vertex $z$ / 10 cm',
)
# AV
ax.add_patch(matplotlib.patches.Rectangle(
    (-894.95, -181.86), 894.95*2, 134.96 - (-181.86), **av_rect_kwargs
))
# NuMI dir
draw_numi_line(ax, numi_dz, numi_dy, anchor_x=0., anchor_y=0.)

##### legend
legend_handles = [
  matplotlib.lines.Line2D(
    [0], [0], color='black',
    lw=1.25, label='ICARUS AV'
  ),
  matplotlib.lines.Line2D(
    [0], [0], color='red',
    lw=1.25, linestyle='-',
    label='NuMI direction'
  ),
]
ax.legend(handles=legend_handles, loc='upper right',)

##### shared colorbar
cbar = fig.colorbar(
    h2d_xy[3],
    ax=axes,
    location='right',
    pad=0.02,
)

cbar.set_label('counts [#]')

plt.show()
fig.savefig(f"plots/nocut/nocut_truth_dirtonly_{var}.pdf", dpi=300)

In [ ]:
pot_dirt

#### Pre-selection

After the not-clear-cosmic cut, vertex in fiducial volume cuts, and after the flash matching cut. 


In [ ]:
FILE_CV = "preselection/CNAF_CV_1eNp0pi_NuMI_NoSysts_Preselection.root"
FILE_OFFBEAM = "preselection/CNAF_OffBeam_1eNp0pi_NuMI_NoSysts_Preselection.root"
FILE_DATA = "preselection/CNAF_Data_1eNp0pi_NuMI_NoSysts_Preselection.root"

In [ ]:
# MC
df = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedNu"  
)
pot = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedCos"  
)
pot_cos = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos['cosmic'] = 1
df = pandas.concat(
  (df, df_cos)
)

# data
df_data = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_DATA}",
  "events/selectedData"  
)
pot_data = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_DATA}")
time_data = sbruceana.utils.get_livetime_data(f"{PATH_TO_SBRUCE}{FILE_DATA}")

# offbeam
df_offbeam = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_OFFBEAM}",
  "offbeam/selectedOffbeam"  
)
time_offbeam = sbruceana.utils.get_livetime_offbeam(f"{PATH_TO_SBRUCE}{FILE_OFFBEAM}")

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "deltaZ_Trigger"
width = 2.5; bins = numpy.arange(0., 100+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED)

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = '|$z_\\mathrm{light}$ - $z_\\mathrm{charge}$| [cm]',
  ylabel = f'slices [{tag_count}]\n/ {width} cm',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=8.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/preselection/preselection_{var}.pdf", dpi=300)

In [ ]:
print("Cosmic background rejection after pre-selection: ", 100 - 100 * (2277.36+8889.20) / (127514.78 + 288982.15))
print("Cosmic background rejection after pre-selection: ", 100 - (9.27 + 36.20))

#### Electron identification

After identifying a shower particle with NuGraph2, plot its shower energy on Collection with no cuts.

In [ ]:
FILE_CV = "preselection_electron_exists/CNAF_CV_1eNp0pi_NuMI_NoSysts_PreselectionelectronExists.root"
FILE_OFFBEAM = "preselection_electron_exists/CNAF_OffBeam_1eNp0pi_NuMI_NoSysts_PreselectionElectronExists.root"
FILE_DATA = "preselection_electron_exists/CNAF_Data_1eNp0pi_NuMI_NoSysts_PreselectionelectronExists.root"

In [ ]:
# MC
df = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedNu"  
)
pot = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedCos"  
)
pot_cos = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos['cosmic'] = 1
df = pandas.concat(
  (df, df_cos)
)

# data
df_data = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_DATA}",
  "events/selectedData"  
)
pot_data = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_DATA}")
time_data = sbruceana.utils.get_livetime_data(f"{PATH_TO_SBRUCE}{FILE_DATA}")

# offbeam
df_offbeam = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_OFFBEAM}",
  "offbeam/selectedOffbeam"  
)
time_offbeam = sbruceana.utils.get_livetime_offbeam(f"{PATH_TO_SBRUCE}{FILE_OFFBEAM}")

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "collE"
width = 0.05; bins = numpy.arange(0., 1+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED)

ax = sbruceana.plotting.place_cut(ax, 0.2, False)

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'leading shower $E$ [GeV]',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=9.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/preselection_electron_exists/preselection_electron_exists_{var}_{int(IS_AREA_NORMALIZED)}.pdf", dpi=300)

#### Electron quality cuts

After cutting on the energy >200 MeV, show the quality of the leading shower along with the cuts.

In [ ]:
FILE_CV = "preselection_electron/CNAF_CV_1eNp0pi_NuMI_NoSysts_Preselectionelectron.root"
FILE_OFFBEAM = "preselection_electron/CNAF_OffBeam_1eNp0pi_NuMI_NoSysts_PreselectionElectron.root"
FILE_DATA = "preselection_electron/CNAF_Data_1eNp0pi_NuMI_NoSysts_Preselectionelectron.root"

In [ ]:
# MC
df = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedNu"  
)
pot = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedCos"  
)
pot_cos = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos['cosmic'] = 1
df = pandas.concat(
  (df, df_cos)
)

# data
df_data = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_DATA}",
  "events/selectedData"  
)
pot_data = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_DATA}")
time_data = sbruceana.utils.get_livetime_data(f"{PATH_TO_SBRUCE}{FILE_DATA}")

# offbeam
df_offbeam = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_OFFBEAM}",
  "offbeam/selectedOffbeam"  
)
time_offbeam = sbruceana.utils.get_livetime_offbeam(f"{PATH_TO_SBRUCE}{FILE_OFFBEAM}")

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "collE"
width = 0.1; bins = numpy.arange(0.2, 2+width, width)

ax = sbruceana.plotting.plot_by_category(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, 1, 1, True, True)
ax = sbruceana.plotting.plot_data(ax, df_offbeam, bins, var, True)

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'leading shower $E$ [GeV]',
  ylabel = f'slices [a.n.]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10, title='NuMI CV'); leg.get_title().set_fontsize(11)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "collE"
width = 0.1; bins = numpy.arange(0.2, 2+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam, yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED)

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'leading shower $E$ [GeV]',
  ylabel = f'slices [{tag_count}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=9); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/preselection/preselection_electron_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "colldEdx"

width = 0.3; bins = numpy.arange(0., 13+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data / time_offbeam , yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data*1.14, bins, var, IS_AREA_NORMALIZED)

ax = sbruceana.plotting.place_cut(ax, 3.5, True)

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'start d$E$/d$x$ [MeV/cm]',
  ylabel = f'slices [{tag_count}]\n/ {width} MeV/cm',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=9); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/preselection/preselection_electron_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "colldEdx"

width = 0.3; bins = numpy.arange(0.3, 13.3+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES_REDUX, bins, var, df_offbeam, offbeam_scale=time_data / time_offbeam , yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data*1.14, bins, var, IS_AREA_NORMALIZED)

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'start d$E$/d$x$ [MeV/cm]',
  ylabel = f'interactions [{tag_count}]\n/ {width} MeV/cm',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=9.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/preselection/ForVulcano_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.25), layout='constrained')

var = "convgap"

width = 0.5; bins = numpy.arange(0., 15+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data/time_offbeam , yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED)

ax = sbruceana.plotting.place_cut(ax, 5, True)

# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'conversion gap [cm]',
  ylabel = f'slices [{tag_count}]\n/ {width} cm',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(ncol=1, fontsize=9); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/preselection/preselection_electron_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.25), layout='constrained')

var = "openangle"

width = 1.; bins = numpy.arange(0., 30+width, width)

ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data / time_offbeam , yscale=pot_data/pot, area_normalized=False, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, False)

ax = sbruceana.plotting.place_cut(ax, 10, True)


# gfx
tag_count = '#'
if IS_AREA_NORMALIZED:
  tag_count = 'a.n.'
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'opening angle [deg.]',
  ylabel = f'slices [{tag_count}]\n/ {width} deg.',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=9); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/preselection/preselection_electron_{var}.pdf", dpi=300)

In [ ]:
fig, axes = plt.subplots(figsize=(4*2, 3*2), ncols=2, nrows=2, layout='constrained')

IS_AREA_NORMALIZED = True
COUNT_TAG = '#'
if IS_AREA_NORMALIZED:
  COUNT_TAG = 'a.n.'

# shower energy
ax = axes[0, 0]

var = "collE"
width = 0.1; bins = numpy.arange(0.2, 2+width, width)
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data / time_offbeam , yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED)

# gfx
ax.set(
  # title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'leading shower $E$ [GeV]',
  ylabel = f'slices [{COUNT_TAG}]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=9, title='NuMI CV'); leg.get_title().set_fontsize(9.5)
leg = ax.legend(fontsize=8.3)

# dEdx
ax = axes[0, 1]

var = "colldEdx"
width = 0.3; bins = numpy.arange(0., 13+width, width)
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data / time_offbeam , yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data*1.14, bins, var, IS_AREA_NORMALIZED)
ax = sbruceana.plotting.place_cut(ax, 3.5, True)

# gfx
ax.set(
  # title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'start d$E$/d$x$ [MeV/cm]',
  ylabel = f'slices [{COUNT_TAG}]\n/ {width} MeV/cm',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=8.5)

# gap
ax = axes[1, 0]

var = "convgap"
width = 0.5; bins = numpy.arange(0., 15+width, width)
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data / time_offbeam , yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED)
ax = sbruceana.plotting.place_cut(ax, 5, True)

# gfx
ax.set(
  # title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'conversion gap [cm]',
  ylabel = f'slices [{COUNT_TAG}]\n/ {width} cm',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=8.5)

# opening angle
ax = axes[1, 1]

var = "openangle"
width = 1.; bins = numpy.arange(0., 30+width, width)
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data / time_offbeam , yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED)
ax = sbruceana.plotting.place_cut(ax, 10, True)

# gfx
ax.set(
  # title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'opening angle [deg.]',
  ylabel = f'slices [{COUNT_TAG}]\n/ {width} deg.',
  xlim   = (bins[0], bins[-1]),
)
# leg = ax.legend(fontsize=8.5)

fig.suptitle(f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT', x=0.98, ha='right', fontsize=13, c='gray')
# fig.suptitle(f'After electron-ID', x=0.02, ha='left', fontsize=13, c='black')

plt.show()
fig.savefig(f"plots/preselection/preselection_electron.pdf", dpi=300)

#### Proton identification + other vetos

After identifying N protons, show for example the number of protons, and the momentum of the leading protons.

Then, explain reason behind vetos, and show final neutrino properties.

Eventually, calibrate energies.

In [ ]:
FILE_CV = "final/CNAF_CV_1eNp0pi_NuMI_NoSysts_FinalSelection.root"
FILE_OFFBEAM = "final/CNAF_OffBeam_1eNp0pi_NuMI_NoSysts_FinalSelection.root"
FILE_DATA = "final/CNAF_Data_1eNp0pi_NuMI_NoSysts_FinalSelection.root"

In [ ]:
# MC
df = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedNu"  
)
pot = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedCos"  
)
pot_cos = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos['cosmic'] = 1
df = pandas.concat(
  (df, df_cos)
)

# data
df_data = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_DATA}",
  "events/selectedData"  
)
pot_data = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_DATA}")
time_data = sbruceana.utils.get_livetime_data(f"{PATH_TO_SBRUCE}{FILE_DATA}")

# offbeam
df_offbeam = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_OFFBEAM}",
  "offbeam/selectedOffbeam"  
)
time_offbeam = sbruceana.utils.get_livetime_offbeam(f"{PATH_TO_SBRUCE}{FILE_OFFBEAM}")

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

IS_AREA_NORMALIZED = True
COUNT_TAG = '#'
if IS_AREA_NORMALIZED:
  COUNT_TAG = 'a.n.'

var = "recoE"
# width = 0.15; bins = numpy.arange(0.2, 2.5+width, width)
# bins = numpy.array([0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1., 1.1, 1.2, 1.3, 1.4, 1.5, 2, 3])
bins = numpy.array([0.2, 0.4, 0.6, 0.8, 1., 1.25, 1.5, 1.75, 2, 2.5, 3])

ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data / time_offbeam , yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED)

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = '$E_{\\nu}$ [GeV]',
  ylabel = f'slices [{COUNT_TAG}]',#\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=9); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "convgap"
width = 1; bins = numpy.arange(0, 5+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data / time_offbeam , yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED)

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'conversion gap [cm]',
  ylabel = f'slices [a.n.]',#\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(ncol=2, fontsize=8.); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "deltapt"
# width = 0.2; bins = numpy.arange(0., 2.+width, width)
bins = numpy.array([0, 0.2, 0.4, 0.6, 0.8, 1, 1.25, 1.5, 2])

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data / time_offbeam , yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data*0.86, bins, var, IS_AREA_NORMALIZED)

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = '$P_{T}$ [GeV]',
  ylabel = f'slices [a.n.]',#\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=9.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "colldEdx"
width = 0.3; bins = numpy.arange(0.2, 8+width, width)

ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data / time_offbeam , yscale=pot_data/pot, area_normalized=False, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, False)

ax = sbruceana.plotting.place_cut(ax, 3.5, True)

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'start d$E$/d$x$ [MeV/cm]',
  ylabel = f'slices [#]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=8); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_{var}.pdf", dpi=300)

In [ ]:
FACTOR = 3e20 / pot
FACTOR * 1447, FACTOR * 284, FACTOR * 120, FACTOR * 47

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "collE"
width = 0.15; bins = numpy.arange(0.2, 2+width, width)

ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data / time_offbeam , yscale=pot_data/pot, area_normalized=False, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, False)

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'leading shower $E$ [GeV]',
  ylabel = f'slices [#]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=8); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "leadpmom"
width = 0.2; bins = numpy.arange(0.2, 2+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data / time_offbeam , yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED)

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'leading proton $p$ [GeV]',
  ylabel = f'slices [a.n.]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=9.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "subleadpmom"
width = 0.2; bins = numpy.arange(0.2, 2+width, width)

IS_AREA_NORMALIZED = True
ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data / time_offbeam , yscale=pot_data/pot, area_normalized=IS_AREA_NORMALIZED, band=True, clip=False)
ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED)

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'sub-leading proton $p$ [GeV]',
  ylabel = f'slices [#]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=8); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/final/final_{var}.pdf", dpi=300)